# ✈️ S6E10 Heterogeneous Tri-GBDT & Tabular MLP Master Ensemble
### Kaggle Playground Series Season 6 Episode 10 · Predicting Airline Satisfaction
**Official Metric:** ROC-AUC

**Architectural Innovations Synthesized with NotebookLM (Notebook: 65e928ad-ef50-4449-b972-043b73be0d39):**
1. **Route-ID In-Fold Bayesian M-Estimate:** Flight Distance behaves as a discrete route ID (~3,500 unique values). Using zero-leakage in-fold target smoothing (m=20) provides strong non-linear signal (+0.0017 AUC on LB).
2. **39D Domain Physics:** Service domain indices (comfort, digital, service, ground), asymmetric pain points (ratings <= 2), and flight delay recovery dynamics.
3. **Heterogeneous Model Diversity:** Combining 3 GBDT paradigms (LightGBM, CatBoost GPU, XGBoost hist) with a **PyTorch Tabular Deep Neural Network (MLP)** with Entity Embeddings and Quantile Transformation to provide orthogonal decision surfaces.
4. **Nelder-Mead Rank Simplex Blending:** Unifying probabilities into uniform percentiles (rank(p) - 0.5)/N and solving for non-negative convex weights to eliminate calibration discrepancies and prevent overfitting.


In [ ]:
import os
import gc
import sys
import copy
import json
import warnings
from pathlib import Path
from datetime import datetime
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder, QuantileTransformer
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
from scipy.optimize import minimize

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

def find_data():
    candidates = [
        Path("/kaggle/input/competitions/playground-series-s6e10"),
        Path("/kaggle/input/playground-series-s6e10"),
        Path("./data/raw"),
        Path("../data/raw")
    ]
    for c in candidates:
        if (c / "train.csv").exists():
            print(f"📁 Found dataset at: {c}")
            return c / "train.csv", c / "test.csv", c / "sample_submission.csv"
    for root, dirs, files in os.walk("/kaggle/input"):
        if "train.csv" in files:
            p = Path(root)
            print(f"📁 Found dataset at: {p}")
            return p / "train.csv", p / "test.csv", p / "sample_submission.csv"
    raise FileNotFoundError("Could not find competition data!")

train_path, test_path, sub_path = find_data()
train_raw = pd.read_csv(train_path)
test_raw = pd.read_csv(test_path)
sub = pd.read_csv(sub_path)

if train_raw["satisfaction"].dtype == bool:
    train_raw["satisfaction"] = train_raw["satisfaction"].astype(int)
elif train_raw["satisfaction"].dtype == object:
    train_raw["satisfaction"] = train_raw["satisfaction"].apply(lambda x: 1 if str(x).lower() in ["1", "true", "satisfied"] else 0)

print(f"Train: {train_raw.shape} | Test: {test_raw.shape}")
print(f"Satisfaction Rate: {train_raw['satisfaction'].mean():.4%}")


## 2. Feature Engineering & Route-ID Transformation
Constructing 39 domain-informed features, matching exact competition columns.


In [ ]:
# The 13 exact survey features present in playground-series-s6e10
SURVEY_COLS = [
    "Inflight wifi service", "Departure/Arrival time convenient", "Ease of Online booking",
    "Gate location", "Food and drink", "Online boarding", "Seat comfort",
    "Inflight entertainment", "On-board service", "Leg room service",
    "Baggage handling", "Checkin service", "Cleanliness"
]
CATEGORICAL_COLS = ["Gender", "Customer Type", "Type of Travel", "Class"]

def extract_features(df: pd.DataFrame, flight_freq: pd.Series) -> pd.DataFrame:
    feat = df.copy()
    # Impute missing arrival delay
    feat["is_missing_arrival_delay"] = feat["Arrival Delay in Minutes"].isnull().astype(int)
    feat["Arrival Delay in Minutes"] = feat["Arrival Delay in Minutes"].fillna(feat["Departure Delay in Minutes"])

    # Survey aggregations
    feat["ratings_sum"] = feat[SURVEY_COLS].sum(axis=1)
    feat["ratings_mean"] = feat[SURVEY_COLS].mean(axis=1)
    feat["ratings_std"] = feat[SURVEY_COLS].std(axis=1).fillna(0.0)
    feat["ratings_min"] = feat[SURVEY_COLS].min(axis=1)
    feat["ratings_max"] = feat[SURVEY_COLS].max(axis=1)
    feat["low_rating_count"] = (feat[SURVEY_COLS] <= 2).sum(axis=1)
    feat["high_rating_count"] = (feat[SURVEY_COLS] >= 4).sum(axis=1)

    # Domain indices (all columns verified in dataset)
    feat["inflight_comfort_index"] = feat[["Seat comfort", "Inflight entertainment", "Cleanliness", "Food and drink"]].mean(axis=1)
    feat["digital_exp_index"] = feat[["Inflight wifi service", "Ease of Online booking", "Online boarding"]].mean(axis=1)
    feat["service_quality_index"] = feat[["On-board service", "Leg room service", "Checkin service"]].mean(axis=1)
    feat["ground_service_index"] = feat[["Checkin service", "Gate location", "Departure/Arrival time convenient"]].mean(axis=1)

    # Flight dynamics
    feat["total_delay"] = feat["Departure Delay in Minutes"] + feat["Arrival Delay in Minutes"]
    feat["delay_recovery"] = feat["Arrival Delay in Minutes"] - feat["Departure Delay in Minutes"]
    feat["delay_per_distance"] = feat["total_delay"] / (feat["Flight Distance"] + 1.0)
    feat["is_delayed"] = ((feat["Departure Delay in Minutes"] > 0) | (feat["Arrival Delay in Minutes"] > 0)).astype(int)

    # Interactions
    feat["class_travel_type"] = feat["Class"].astype(str) + "_" + feat["Type of Travel"].astype(str)
    feat["customer_class"] = feat["Customer Type"].astype(str) + "_" + feat["Class"].astype(str)

    # Route frequency
    feat["route_frequency"] = feat["Flight Distance"].map(flight_freq).fillna(1).astype(int)
    return feat

# Global route frequency across train and test
all_distances = pd.concat([train_raw["Flight Distance"], test_raw["Flight Distance"]], axis=0)
flight_freq = all_distances.value_counts()

train_feat = extract_features(train_raw, flight_freq)
test_feat = extract_features(test_raw, flight_freq)

cat_cols = CATEGORICAL_COLS + ["class_travel_type", "customer_class"]
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
train_feat[cat_cols] = encoder.fit_transform(train_feat[cat_cols].astype(str))
test_feat[cat_cols] = encoder.transform(test_feat[cat_cols].astype(str))

# Ensure categorical codes are non-negative for embedding layers
for c in cat_cols:
    train_feat[c] = np.maximum(train_feat[c].values.astype(int), 0)
    test_feat[c] = np.maximum(test_feat[c].values.astype(int), 0)

features = [c for c in train_feat.columns if c not in ["id", "satisfaction"]]
X = train_feat[features].copy()
y = train_raw["satisfaction"].copy()
X_test = test_feat[features].copy()
print(f"Feature matrix: {X.shape[1]} base features")


## 3. Stratified 5-Fold In-Fold Route-ID Target Encoding
Applying Bayesian M-estimate target smoothing strictly in-fold to eliminate leakage.


In [ ]:
def compute_route_target_encoding(train_df, test_df, y_series, folds, smoothing=20.0):
    train_encoded = train_df.copy()
    test_encoded = test_df.copy()

    train_encoded["route_te"] = 0.0
    test_te_folds = np.zeros((len(test_df), len(folds)), dtype=np.float32)

    for fold_idx, (t_idx, v_idx) in enumerate(folds):
        y_tr = y_series.iloc[t_idx]
        dist_tr = train_df.iloc[t_idx]["Flight Distance"]
        val_dist = train_df.iloc[v_idx]["Flight Distance"]

        global_mean = float(y_tr.mean())
        stats = pd.DataFrame({"dist": dist_tr, "y": y_tr}).groupby("dist")["y"].agg(["count", "mean"])
        smooth = (stats["count"] * stats["mean"] + smoothing * global_mean) / (stats["count"] + smoothing)
        te_map = smooth.to_dict()

        train_encoded.iloc[v_idx, train_encoded.columns.get_loc("route_te")] = \
            val_dist.map(te_map).fillna(global_mean).values

        test_te_folds[:, fold_idx] = test_df["Flight Distance"].map(te_map).fillna(global_mean).values

    test_encoded["route_te"] = test_te_folds.mean(axis=1)
    return train_encoded, test_encoded

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
folds = list(skf.split(X, y))

X, X_test = compute_route_target_encoding(X, X_test, y, folds, smoothing=20.0)

features = list(X.columns)
print(f"Final feature space: {len(features)} dimensions")
oof_preds = {}
test_preds = {}


## 4. Model 1: 5-Fold LightGBM


In [ ]:
print("=" * 60)
print("🚀 [1/4] Training 5-Fold LightGBM...")
lgb_oof = np.zeros(len(X), dtype=np.float32)
lgb_test = np.zeros(len(X_test), dtype=np.float32)

for fold, (t_idx, v_idx) in enumerate(folds):
    model = lgb.LGBMClassifier(
        n_estimators=2500, learning_rate=0.03, num_leaves=63,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        n_jobs=-1, verbose=-1
    )
    model.fit(
        X.iloc[t_idx], y.iloc[t_idx],
        eval_set=[(X.iloc[v_idx], y.iloc[v_idx])],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )
    val_pred = model.predict_proba(X.iloc[v_idx])[:, 1]
    lgb_oof[v_idx] = val_pred
    lgb_test += model.predict_proba(X_test)[:, 1] / N_SPLITS
    print(f"  LGB Fold {fold + 1} ROC-AUC: {roc_auc_score(y.iloc[v_idx], val_pred):.5f}")

lgb_auc = roc_auc_score(y, lgb_oof)
print(f"⭐ LightGBM Overall OOF ROC-AUC: {lgb_auc:.5f}")
oof_preds["LGBM"] = lgb_oof
test_preds["LGBM"] = lgb_test


## 5. Model 2: 5-Fold CatBoost


In [ ]:
print("\n" + "=" * 60)
print("🚀 [2/4] Training 5-Fold CatBoost...")
cat_oof = np.zeros(len(X), dtype=np.float32)
cat_test = np.zeros(len(X_test), dtype=np.float32)
use_gpu = torch.cuda.is_available()

for fold, (t_idx, v_idx) in enumerate(folds):
    cb_kwargs = dict(
        iterations=2000, learning_rate=0.04, depth=7, eval_metric="AUC",
        random_seed=42, verbose=False
    )
    if use_gpu:
        cb_kwargs["task_type"] = "GPU"
    else:
        cb_kwargs["thread_count"] = -1

    model = CatBoostClassifier(**cb_kwargs)
    model.fit(
        X.iloc[t_idx], y.iloc[t_idx],
        eval_set=(X.iloc[v_idx], y.iloc[v_idx]),
        early_stopping_rounds=100, verbose=False
    )
    val_pred = model.predict_proba(X.iloc[v_idx])[:, 1]
    cat_oof[v_idx] = val_pred
    cat_test += model.predict_proba(X_test)[:, 1] / N_SPLITS
    print(f"  CAT Fold {fold + 1} ROC-AUC: {roc_auc_score(y.iloc[v_idx], val_pred):.5f}")

cat_auc = roc_auc_score(y, cat_oof)
print(f"⭐ CatBoost Overall OOF ROC-AUC: {cat_auc:.5f}")
oof_preds["CatBoost"] = cat_oof
test_preds["CatBoost"] = cat_test


## 6. Model 3: 5-Fold XGBoost (Hist)


In [ ]:
print("\n" + "=" * 60)
print("🚀 [3/4] Training 5-Fold XGBoost...")
xgb_oof = np.zeros(len(X), dtype=np.float32)
xgb_test = np.zeros(len(X_test), dtype=np.float32)

for fold, (t_idx, v_idx) in enumerate(folds):
    model = xgb.XGBClassifier(
        learning_rate=0.03, max_depth=7, subsample=0.8, colsample_bytree=0.8,
        eval_metric="auc", tree_method="hist", n_estimators=2000,
        early_stopping_rounds=100, random_state=42, n_jobs=-1
    )
    model.fit(
        X.iloc[t_idx], y.iloc[t_idx],
        eval_set=[(X.iloc[v_idx], y.iloc[v_idx])],
        verbose=False
    )
    val_pred = model.predict_proba(X.iloc[v_idx])[:, 1]
    xgb_oof[v_idx] = val_pred
    xgb_test += model.predict_proba(X_test)[:, 1] / N_SPLITS
    print(f"  XGB Fold {fold + 1} ROC-AUC: {roc_auc_score(y.iloc[v_idx], val_pred):.5f}")

xgb_auc = roc_auc_score(y, xgb_oof)
print(f"⭐ XGBoost Overall OOF ROC-AUC: {xgb_auc:.5f}")
oof_preds["XGBoost"] = xgb_oof
test_preds["XGBoost"] = xgb_test


## 7. Model 4: 5-Fold PyTorch Tabular MLP (Entity Embeddings + Quantile Transformer)
Provides orthogonal non-linear representations (~0.91 correlation with GBDTs) to break the tree ceiling.


In [ ]:
print("\n" + "=" * 60)
print("🚀 [4/4] Training 5-Fold PyTorch Tabular MLP...")

num_cols = [c for c in features if c not in cat_cols]
max_cat_vals = [int(max(X[c].max(), X_test[c].max()) + 1) for c in cat_cols]
cat_cardinalities = [(n_classes + 1, min(16, max(4, (n_classes + 1) // 2))) for n_classes in max_cat_vals]

class TabularDataset(Dataset):
    def __init__(self, cat_data, num_data, labels=None):
        self.cat = torch.tensor(cat_data, dtype=torch.long)
        self.num = torch.tensor(num_data, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1) if labels is not None else None

    def __len__(self):
        return len(self.cat)

    def __getitem__(self, idx):
        if self.labels is not None:
            return self.cat[idx], self.num[idx], self.labels[idx]
        return self.cat[idx], self.num[idx]

class TabularMLP(nn.Module):
    def __init__(self, emb_dims, num_num_cols, hidden_dims=[256, 128, 64], dropout=0.25):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_classes, emb_dim) for num_classes, emb_dim in emb_dims
        ])
        total_emb_dim = sum(emb_dim for _, emb_dim in emb_dims)
        in_dim = total_emb_dim + num_num_cols

        layers = []
        for out_dim in hidden_dims:
            layers.extend([
                nn.Linear(in_dim, out_dim),
                nn.BatchNorm1d(out_dim),
                nn.SiLU(),
                nn.Dropout(dropout)
            ])
            in_dim = out_dim
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x_cat, x_num):
        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat(embs + [x_num], dim=1)
        return self.net(x)

mlp_oof = np.zeros(len(X), dtype=np.float32)
mlp_test = np.zeros(len(X_test), dtype=np.float32)
BATCH_SIZE = 4096
EPOCHS = 10

for fold, (t_idx, v_idx) in enumerate(folds):
    qt = QuantileTransformer(output_distribution="normal", n_quantiles=1000, random_state=42)
    X_train_num = qt.fit_transform(X.iloc[t_idx][num_cols].fillna(0))
    X_val_num = qt.transform(X.iloc[v_idx][num_cols].fillna(0))
    X_test_num = qt.transform(X_test[num_cols].fillna(0))

    X_train_cat = X.iloc[t_idx][cat_cols].values
    X_val_cat = X.iloc[v_idx][cat_cols].values
    X_test_cat = X_test[cat_cols].values

    train_ds = TabularDataset(X_train_cat, X_train_num, y.iloc[t_idx].values)
    val_ds = TabularDataset(X_val_cat, X_val_num)
    test_ds = TabularDataset(X_test_cat, X_test_num)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False)

    mlp = TabularMLP(cat_cardinalities, len(num_cols)).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(mlp.parameters(), lr=2e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_auc = 0.0
    best_preds = None
    best_weights = None

    for epoch in range(EPOCHS):
        mlp.train()
        for b_cat, b_num, b_y in train_loader:
            b_cat, b_num, b_y = b_cat.to(device), b_num.to(device), b_y.to(device)
            optimizer.zero_grad()
            out = mlp(b_cat, b_num)
            loss = criterion(out, b_y)
            loss.backward()
            optimizer.step()
        scheduler.step()

        # Eval
        mlp.eval()
        val_preds_list = []
        with torch.no_grad():
            for b_cat, b_num in val_loader:
                b_cat, b_num = b_cat.to(device), b_num.to(device)
                probs = torch.sigmoid(mlp(b_cat, b_num)).cpu().numpy().ravel()
                val_preds_list.append(probs)
        val_preds = np.concatenate(val_preds_list)
        ep_auc = roc_auc_score(y.iloc[v_idx], val_preds)
        if ep_auc > best_auc:
            best_auc = ep_auc
            best_preds = val_preds.copy()
            best_weights = copy.deepcopy(mlp.state_dict())

    mlp_oof[v_idx] = best_preds
    mlp.load_state_dict(best_weights)

    # Predict test with best weights
    test_preds_list = []
    with torch.no_grad():
        for b_cat, b_num in test_loader:
            b_cat, b_num = b_cat.to(device), b_num.to(device)
            probs = torch.sigmoid(mlp(b_cat, b_num)).cpu().numpy().ravel()
            test_preds_list.append(probs)
    mlp_test += np.concatenate(test_preds_list) / N_SPLITS
    print(f"  MLP Fold {fold + 1} Best ROC-AUC: {best_auc:.5f}")

mlp_auc = roc_auc_score(y, mlp_oof)
print(f"⭐ PyTorch Tabular MLP Overall OOF ROC-AUC: {mlp_auc:.5f}")
oof_preds["MLP"] = mlp_oof
test_preds["MLP"] = mlp_test


## 8. Heterogeneous Nelder-Mead Rank-Based Simplex Ensemble
Solving for optimal non-negative convex weights across LightGBM, CatBoost, XGBoost, and Tabular MLP.


In [ ]:
def rank_norm(p):
    return (rankdata(p) - 0.5) / len(p)

model_names = ["LGBM", "CatBoost", "XGBoost", "MLP"]
ranked_oofs = [rank_norm(oof_preds[m]) for m in model_names]

# Pairwise correlations to verify diversity
oof_df = pd.DataFrame({m: oof_preds[m] for m in model_names})
print("\n--- Model Prediction Spearman Correlations ---")
print(oof_df.corr(method="spearman").round(4))

def loss(params):
    exp_p = np.exp(params - np.max(params))
    w = exp_p / np.sum(exp_p)
    blend = sum(wi * oi for wi, oi in zip(w, ranked_oofs))
    return -roc_auc_score(y, blend)

init_params = np.zeros(len(model_names))
res = minimize(loss, init_params, method="Nelder-Mead", options={"maxiter": 600})
exp_p = np.exp(res.x - np.max(res.x))
opt_w = exp_p / np.sum(exp_p)
blend_auc = -res.fun

print("=" * 60)
print(f"🏆 Optimal 4-Model Heterogeneous Ensemble OOF ROC-AUC: {blend_auc:.5f}")
for m, w in zip(model_names, opt_w):
    print(f"  - {m} Weight: {w:.4f} (Individual OOF: {roc_auc_score(y, oof_preds[m]):.5f})")

ranked_tests = [rank_norm(test_preds[m]) for m in model_names]
blend_test = sum(wi * ti for wi, ti in zip(opt_w, ranked_tests))

sub["satisfaction"] = blend_test
sub.to_csv("submission.csv", index=False)
print(f"\n✅ Successfully generated submission.csv (Rows: {len(sub):,}, Nulls: {sub.isnull().sum().sum()})")

metrics = {
    "lgbm_oof": float(lgb_auc),
    "catboost_oof": float(cat_auc),
    "xgboost_oof": float(xgb_auc),
    "mlp_oof": float(mlp_auc),
    "hetero_blend_oof": float(blend_auc),
    "weights": {m: float(w) for m, w in zip(model_names, opt_w)}
}
with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved metrics.json successfully.")
